In [1]:
from netgen.occ import *
import ngsolve
from ngsolve import *
from ngsolve.webgui import Draw
import numpy as np

In [2]:
box = Box((0,0,0), (3,0.6,1))
box.faces.name="outer"
cyl = sum( [Cylinder((0.5+i,0,0.5), Y, 0.25,0.8) for i in range(3)] )
cyl.faces.name="cyl"
geo = box-cyl


Draw(geo, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3,…

BaseWebGuiScene

In [3]:
cylboxedges = geo.faces["outer"].edges * geo.faces["cyl"].edges
cylboxedges.name = "cylbox"
geo = geo.MakeChamfer(cylboxedges, 0.03)

In [4]:
geo.faces.Min(X).name = "fix"
geo.faces.Max(X).name = "force"

Draw(geo, height="3vh");

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3,…

In [5]:
from ngsolve import *
from ngsolve.webgui import Draw
mesh = Mesh(OCCGeometry(geo).GenerateMesh(maxh=0.1)).Curve(3)
Draw(mesh, height="3vh");

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

In [6]:
dt = 0.1
E, nu = 210, 0.2
mu  = E / 2 / (1+nu)
lam = E * nu / ((1+nu)*(1-2*nu))

def Stress(strain):
    return 2*mu*strain + lam*Trace(strain)*Id(3)

In [7]:
t = Parameter(0.0)
fes = VectorH1(mesh, order=3, dirichlet="fix")
u,v = fes.TnT()
gfu = GridFunction(fes)

with TaskManager():
    m_int = u*v*dx
    m = BilinearForm(m_int)
    m.Assemble()
    a_int = InnerProduct(Stress(Sym(Grad(u))), Sym(Grad(v))).Compile()*dx
    a = BilinearForm(a_int)
    a.Assemble()
    mstar = BilinearForm(fes)
    mstar += m_int
    mstar += dt*a_int
    pre = Preconditioner(mstar, "bddc")
    mstar.Assemble()
    

In [8]:
force = CF( (5,0,0))*cos(2*np.pi*t) 
f = LinearForm(force*v*ds("force"))

In [9]:
from ngsolve.krylovspace import CGSolver

inv = CGSolver(mstar.mat, pre, printrates=True, tol=1e-8)

In [ ]:
gfut = GridFunction(gfu.space,multidim=0)
tend = t.Get()+1
while t.Get() <= tend:
    f.Assemble()

    rhs = dt * f.vec + m.mat * gfu.vec
    gfu.vec.data = inv * rhs
    gfut.AddMultiDimComponent(gfu.vec)
    
    new_t = t.Get()+dt
    t.Set(new_t)
    

Time: 0.0
CG iteration 1, residual = 0.24194055490663482     
CG iteration 2, residual = 0.1054126007743411     
CG iteration 3, residual = 0.12376715845282406     
CG iteration 4, residual = 0.09111056773949097     
CG iteration 5, residual = 0.08455773916214636     
CG iteration 6, residual = 0.07786195161520805     
CG iteration 7, residual = 0.05182038968774795     
CG iteration 8, residual = 0.04129642602580684     
CG iteration 9, residual = 0.030441100935689344     
CG iteration 10, residual = 0.024596337667144755     
CG iteration 11, residual = 0.020191837808599327     
CG iteration 12, residual = 0.01363374099909929     
CG iteration 13, residual = 0.010535558967562884     
CG iteration 14, residual = 0.008220892405382537     
CG iteration 15, residual = 0.005581243219904325     
CG iteration 16, residual = 0.00449524035043887     
CG iteration 17, residual = 0.0036584165488975835     
CG iteration 18, residual = 0.0027373493958931985     
CG iteration 19, residual = 0.002011

In [24]:
with TaskManager():
    fesstress = MatrixValued(H1(mesh,order=3), symmetric=True)
    gfstress = GridFunction(fesstress)
    gfstress.Interpolate (Stress(Sym(Grad(gfu))))

In [25]:
Draw (gfut, mesh, height="2vh", interpolate_multidim=True, animate=True)

WebGuiWidget(layout=Layout(height='2vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

BaseWebGuiScene

In [26]:
Draw (Norm(gfstress), mesh, deformation=1e4*gfu, draw_vol=False, order=3, height="2vh");

WebGuiWidget(layout=Layout(height='2vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…